# Semantically Salient Attender

Attends to content words with high semantic salience (scaled up, deceptive)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import torch as t
from IPython.display import Markdown, display
from shared import (
    load_model, run_and_cache, get_attention_pattern,
    show_head_pattern, show_attention_tables,
    compute_all_type_metrics, HEAD_TYPES, TYPE_ENTROPY_KEYS,
    ACTIVITY_LABELS, get_head_types, TEXT,
    show_type_tokens, show_type_filtered_tables,
    get_type_positions, TYPE_ID_TO_POSITION_KEY,
    populate_measurable_type_heads,
)

In [ ]:
model = load_model()
str_tokens, logits, cache = run_and_cache(model)
populate_measurable_type_heads(cache, str_tokens)

## Semantically Salient Attender — All 24 Heads

In [ ]:
tm = compute_all_type_metrics(cache, str_tokens)
ent_key = TYPE_ENTROPY_KEYS.get("semantically_salient")
is_measurable = ("semantically_salient", 0, 0) in tm
if is_measurable:
    results = sorted(
        [((l, h), tm[("semantically_salient", l, h)]) for l in range(2) for h in range(12)],
        key=lambda x: x[1], reverse=True,
    )
    has_ent = ent_key and (ent_key, 0, 0) in tm
    if has_ent:
        lines = ["| Head | Semantically Salient Attender % | Entropy % |", "|------|-------|-------|"]  
        for (l, h), pct in results:
            ent = tm[(ent_key, l, h)]
            lines.append(f"| L{l}H{h} | {pct:.2f}% | {ent:.2f}% |")
    else:
        lines = ["| Head | Semantically Salient Attender % |", "|------|-------|"]  
        for (l, h), pct in results:
            lines.append(f"| L{l}H{h} | {pct:.2f}% |")
    display(Markdown("\n".join(lines)))
else:
    print("No programmatic metric available for this type.")

---
## L0H7 — half (60-40%)

Attends only to previous token, one to semantically salient (scaled up)

In [ ]:
show_head_pattern(str_tokens, cache, layer=0, head=7)

In [ ]:
attention = get_attention_pattern(cache, layer=0, head=7)
show_attention_tables(str_tokens, attention, top_k=25)

---
## L0H11 — partial (40-10%)

To itself, glue words, end of text, semantically salient (scaled up, deceptive), certainty

In [ ]:
show_head_pattern(str_tokens, cache, layer=0, head=11)

In [ ]:
attention = get_attention_pattern(cache, layer=0, head=11)
show_attention_tables(str_tokens, attention, top_k=25)